In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()

/home/mv/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:00<00:00, 2211.63it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [2]:
messages = [
    {"role": "user", "content": "Explain KV cache in simple terms."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print("prompt tokens:", input_ids.shape[1])

prompt tokens: 20


In [3]:
with torch.inference_mode():
    t0 = time.perf_counter()

    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )

    prefill_time = time.perf_counter() - t0

past = out.past_key_values
next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

print("prefill time:", prefill_time)
print("first next token:", tokenizer.decode(next_token[0]))

prefill time: 0.2081071810098365
first next token: The


In [6]:
def print_kv_shapes(past_key_values, max_layers=3):
    print("cache type:", type(past_key_values))

    # New Transformers cache API
    if hasattr(past_key_values, "layers"):
        layers = past_key_values.layers

        for i, layer in enumerate(layers[:max_layers]):
            k = getattr(layer, "keys", None)
            v = getattr(layer, "values", None)

            if k is None or v is None:
                print(f"layer {i}: empty / not initialized")
            else:
                print(f"layer {i}: K={tuple(k.shape)}, V={tuple(v.shape)}")

        return

    # Older Transformers cache API
    if isinstance(past_key_values, (tuple, list)):
        for i, layer in enumerate(past_key_values[:max_layers]):
            k, v = layer[:2]
            print(f"layer {i}: K={tuple(k.shape)}, V={tuple(v.shape)}")

        return

    print("Unknown cache format")
    print(dir(past_key_values))

print_kv_shapes(past)

cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 20, 128), V=(1, 8, 20, 128)
layer 1: K=(1, 8, 20, 128), V=(1, 8, 20, 128)
layer 2: K=(1, 8, 20, 128), V=(1, 8, 20, 128)


1   = batch size

8   = KV heads

20  = sequence length / prompt tokens

128 = head dimension

In [7]:
generated = []
decode_times = []

max_new_tokens = 20

for step in range(max_new_tokens):
    generated.append(next_token)

    attention_mask = torch.cat(
        [
            attention_mask,
            torch.ones((1, 1), device=model.device, dtype=attention_mask.dtype),
        ],
        dim=1,
    )

    with torch.inference_mode():
        t0 = time.perf_counter()

        out = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past,
            use_cache=True,
        )

        decode_time = time.perf_counter() - t0

    decode_times.append(decode_time)

    past = out.past_key_values
    next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    print(f"\nstep {step}")
    print("token:", tokenizer.decode(generated[-1][0]))
    print("decode time:", decode_time)
    print_kv_shapes(past, max_layers=1)


step 0
token: The
decode time: 0.06658392900135368
cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 21, 128), V=(1, 8, 21, 128)

step 1
token:  **
decode time: 0.010637995001161471
cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 22, 128), V=(1, 8, 22, 128)

step 2
token: KV
decode time: 0.010151441994821653
cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 23, 128), V=(1, 8, 23, 128)

step 3
token:  Cache
decode time: 0.010038565000286326
cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 24, 128), V=(1, 8, 24, 128)

step 4
token: **
decode time: 0.009986676974222064
cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 25, 128), V=(1, 8, 25, 128)

step 5
token:  (
decode time: 0.010025701980339363
cache type: <class 'transformers.cache_utils.DynamicCache'>
layer 0: K=(1, 8, 26, 128), V=(1, 8, 26, 128)

step 6
token: Key
decode time: 0.0102503309899

In [8]:
generated_ids = torch.cat(generated, dim=1)

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

The **KV Cache** (Key-Value Cache) is a type of memory cache used to store


In [9]:
print("prompt tokens:", input_ids.shape[1])
print("generated tokens:", generated_ids.shape[1])
print("prefill time:", prefill_time)
print("avg decode time/token:", sum(decode_times) / len(decode_times))
print("tokens/sec:", len(decode_times) / sum(decode_times))

prompt tokens: 20
generated tokens: 20
prefill time: 0.2081071810098365
avg decode time/token: 0.012659957494179253
tokens/sec: 78.98920675363848
